In [1]:
import pandas as pd
import glob
from inference import get_model_available, get_data_academic_risk, predict_academic_risk

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os

os.add_dll_directory('C:\\Program Files\\IBM\\SQLLIB\\BIN')
# conectar a la base de datos IBM Db2
import ibm_db

In [4]:
conn_str = 'DATABASE=' + os.getenv('DATABASE') + ';HOSTNAME=' + os.getenv('HOSTNAME') + ';PORT=' + os.getenv('PORT') + ';PROTOCOL=TCPIP;UID=' + os.getenv('USERNAME_DB') + ';PWD=' + os.getenv('PASSWORD_DB') + ';'
conn = ibm_db.connect(conn_str, '', '')
# conn = ibm_db.connect(os.getenv('DATABASE'), os.getenv('USERNAME_DB'), os.getenv('PASSWORD_DB')) # os.getenv('HOSTNAME'), os.getenv('PORT')

if conn:
    print("Conexión exitosa")
else:
    print("Error al conectar")

Conexión exitosa


In [5]:
anio_base, termino_base = 2025, 2  # Periodo actual
cod_materia_objetivo = 'MATG1058'#'CCPG1042' # 'CCPG1052'  # Materia objetivo

In [6]:
def get_data_from_query(sql_str):
    stmt_select  = ibm_db.exec_immediate(conn, sql_str)
    
    # Fetch all rows
    data_list = []
    result = ibm_db.fetch_assoc(stmt_select)
    while result:
        # print(result)
        data_list.append(result)
        result = ibm_db.fetch_assoc(stmt_select)
    return data_list

In [7]:
modelo, label_encoders, feature_info = get_model_available()


📂 Cargando modelos y configuración...

✅ Modelo seleccionado: Random Forest: ../models/random_forest_model.pkl
✓ Modelo cargado desde '../models/random_forest_model.pkl'


In [8]:
def make_analysis(list_matricula, cod_materia):
    mi_df = get_data_academic_risk(list_matricula, cod_materia, label_encoders)

    print("Debe coinicidir la cantidad:",mi_df.shape[0] == len(list_matricula))
    if mi_df.shape[0] != len(list_matricula):
        print("⚠️ Advertencia: Algunos estudiantes no tienen datos completos para el análisis.")

    results = predict_academic_risk(modelo, feature_info, mi_df)

    stats = results['statistics']
    # print(f"📊 {stats['pred_aprobar']} estudiantes aprobarán ({stats['pct_aprobar']:.2f}%)")
    
    return results, mi_df, stats

### Datos de historico completo desde 2020 a 2022 2S

In [9]:
list_names_files = glob.glob("../data/riesgo_academico/all_*")
df_materias = pd.read_csv("../data/riesgo_academico/dificultad_materia.csv")
df_complete = pd.DataFrame()

In [10]:
for file in list_names_files:
    print("file", file)
    split_file = file.split("_")
    termino = split_file[-1].split(".")[0]
    if termino == "3S":
        continue
    
    df = pd.read_csv(file)

    df["anio"] = split_file[-2]
    df["termino"] = termino
    if not("MATERIA" in df.keys()):
        df = pd.merge(df, df_materias[["CODIGOMATERIA", "MATERIA"]], left_on="COD_MATERIA_ACAD_MO", right_on="CODIGOMATERIA")

    df_complete = pd.concat([df_complete, df], ignore_index=True)

file ../data/riesgo_academico\all_2020_1S.csv
file ../data/riesgo_academico\all_2020_2S.csv
file ../data/riesgo_academico\all_2021_1S.csv
file ../data/riesgo_academico\all_2021_2S.csv
file ../data/riesgo_academico\all_2022_1S.csv
file ../data/riesgo_academico\all_2022_2S.csv
file ../data/riesgo_academico\all_2023_1S.csv
file ../data/riesgo_academico\all_2023_2S.csv
file ../data/riesgo_academico\all_2024_1S.csv
file ../data/riesgo_academico\all_2024_2S.csv
file ../data/riesgo_academico\all_2025_1S.csv
file ../data/riesgo_academico\all_2025_2S.csv


In [11]:
df_complete["COD_ESTUDIANTE"] = df_complete["COD_ESTUDIANTE"].astype(str)

In [12]:
df_complete.shape

(366900, 30)

In [13]:
df_complete

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,FACIL,MODERADA,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL
0,201160178,ACUG1035,AP,1,87,93,"9,00","7,90",53.0,59.0,...,0,2,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
1,201310353,ACUG1035,AP,1,85,89,"8,70","7,90",59.0,61.0,...,0,2,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
2,201313869,ACUG1035,AP,1,86,89,"8,75","7,90",59.0,56.0,...,1,1,0,1,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
3,201507649,ACUG1035,AP,1,83,93,"8,80","7,90",49.0,65.0,...,0,2,2,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
4,201607884,ACUG1035,AP,1,95,95,"9,50","7,90",39.0,72.0,...,1,1,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366895,202300422,TURG2037,AC,1,0,0,"0,00","7,22",27.0,73.0,...,1,0,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,92"
366896,202104683,TURG2037,AC,1,0,0,"0,00","7,22",37.0,65.0,...,0,0,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,54"
366897,202111548,TURG2037,AC,1,0,0,"0,00","7,22",36.0,66.0,...,0,0,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,53"
366898,202103719,TURG2037,AC,1,0,0,"0,00","7,22",33.0,67.0,...,0,0,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,25"


In [14]:
# group by anio and termino
df_complete.groupby(["anio", "termino"]).size() 

anio  termino
2020  1S         35871
      2S         34086
2021  1S         32727
      2S         31286
2022  1S         29507
      2S         28859
2023  1S         28301
      2S         28195
2024  1S         28707
      2S         28488
2025  1S         30264
      2S         30609
dtype: int64

### Agregar carrera a los datos completos de estudiantes

In [15]:
df_carreras_estudiantes = pd.read_csv("../data/riesgo_academico/carreras_estudiantes.csv")

In [16]:
df_carreras_estudiantes["CODESTUDIANTE"] = df_carreras_estudiantes["CODESTUDIANTE"].astype(str)
df_carreras_estudiantes["CODESTUDIANTE"] = df_carreras_estudiantes["CODESTUDIANTE"].str.strip()

In [17]:
# # agrupa los CODESTUDIANTE y CARRERA y cuenta la cantidad de ocurrencias. Con CODESTUDIANTE y CARRERA como columnas separadas
df_carreras_estudiantes_new = df_carreras_estudiantes.groupby(['CODESTUDIANTE', 'CARRERA', 'ANIO']).size().reset_index(name='CANTIDAD').copy()

In [18]:
# Obtener el año máximo por cada combinación de CODESTUDIANTE y CARRERA
df_carreras_estudiantes_max_anio = df_carreras_estudiantes_new.loc[
    df_carreras_estudiantes_new.groupby(['CODESTUDIANTE', 'CARRERA'])['ANIO'].idxmax()
][['CODESTUDIANTE', 'CARRERA', 'ANIO']].reset_index(drop=True)

In [19]:
# Conservar solo una fila por CODESTUDIANTE con el ANIO máximo
df_carreras_estudiantes_max_anio = (
    df_carreras_estudiantes_max_anio
    .sort_values('ANIO', ascending=False)
    .groupby('CODESTUDIANTE', as_index=False)
    .first()
)

In [20]:
df_carreras_estudiantes_max_anio

,CODESTUDIANTE,CARRERA,ANIO
0,198802423,Acuicultura,2020
1,198901423,Electricidad,2020
2,198903122,Arqueología,2024
3,198905267,Acuicultura,2020
4,199002130,Electricidad,2025
...,...,...,...
19694,202590329,Movilidad Nacional,2025
19695,202590337,Movilidad Nacional,2025
19696,202590345,Movilidad Nacional,2025
19697,202590352,Movilidad Nacional,2025


In [21]:
df_carreras_estudiantes_max_anio["CODESTUDIANTE"].value_counts()

CODESTUDIANTE
202590360    1
198802423    1
198901423    1
202590204    1
202590196    1
            ..
199500075    1
199202169    1
199201526    1
199002130    1
198905267    1
Name: count, Length: 19699, dtype: int64

In [22]:
df_carreras_estudiantes_max_anio[df_carreras_estudiantes_max_anio["CODESTUDIANTE"] == "201908803"]

,CODESTUDIANTE,CARRERA,ANIO
8433,201908803,Alimentos,2025


In [23]:
df_complete.shape, df_carreras_estudiantes_max_anio.shape

((366900, 30), (19699, 3))

In [24]:
# agregar a df_complete CARRERA desde df_carreras_estudiantes haciendo merge con COD_ESTUDIANTE de df_complete y CODESTUDIANTE de df_carreras_estudiantes
df_complete = pd.merge(df_complete, df_carreras_estudiantes_max_anio[['CODESTUDIANTE', 'CARRERA']], left_on='COD_ESTUDIANTE', right_on='CODESTUDIANTE', how='inner')
df_complete = df_complete.drop(columns=['CODESTUDIANTE'])

### Estudiantes que estan viendo actualmente la materia objetivo

In [25]:
dir_estudiantes_viendo = '../database/planificacion_aprobacion/estudiantes_actualmente_viendo.sql'


In [26]:
with open(dir_estudiantes_viendo, 'r', encoding='utf-8') as file:
    sql_estudiantes_viendo_base = file.read()

In [27]:
sql_estudiantes_viendo = sql_estudiantes_viendo_base.split("------------")[0]
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('\n', ' ')
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('TT', f"{termino_base}S")
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('AAAA', f"{anio_base}")
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('xxxxx', f"{cod_materia_objetivo}")

In [28]:
sql_estudiantes_viendo

"SELECT COD_ESTUDIANTE, COD_MATERIA_ACAD, tm.NOMBRE, tpa.nombre carrera, hd.*  FROM espol.HISTORIA_ANIO  hd  INNER JOIN espol.TBL_MATERIA tm ON hd.COD_MATERIA_ACAD =tm.CODIGOMATERIA  INNER JOIN espol.TBL_PROGRAMA_ACADEMICO tpa ON tpa.CODCARRERA =hd.COD_CARRERA and tpa.CODDIVISION =hd.COD_DIVISION AND tpa.CODESPECIALIZ =hd.COD_ESPECIALIZ WHERE tm.codigomateria IN ('MATG1058') AND termino='2S' AND ANIO='2025' AND hd.ESTADO_MAT_TOMADA <>'AN' ;  "

In [29]:
# Fetch all rows
data_list = get_data_from_query(sql_estudiantes_viendo)

In [30]:
df_estudiantes_viendo = pd.DataFrame(data_list)[["COD_ESTUDIANTE", "NOMBRE", "CARRERA", "COD_MATERIA_ACAD", "ANIO", "TERMINO", "VEZ_TOMADA"]].copy()

In [31]:
# hacer antes el copy porque si no no sabe si es en el copy o en el original
df_estudiantes_viendo["COD_MATERIA_ACAD"] = df_estudiantes_viendo["COD_MATERIA_ACAD"].astype(str)
df_estudiantes_viendo["COD_MATERIA_ACAD"] = df_estudiantes_viendo["COD_MATERIA_ACAD"].str.strip() 

In [32]:
df_estudiantes_viendo.shape

(36, 7)

In [33]:
df_estudiantes_viendo.head(8)

,COD_ESTUDIANTE,NOMBRE,CARRERA,COD_MATERIA_ACAD,ANIO,TERMINO,VEZ_TOMADA
0,202003976,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1
1,202317301,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1
2,202416137,OPTIMIZACIÓN NUMÉRICA,Estadística,MATG1058,2025,2S,1
3,202407136,OPTIMIZACIÓN NUMÉRICA,Estadística,MATG1058,2025,2S,1
4,202317137,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1
5,202306775,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1
6,202407318,OPTIMIZACIÓN NUMÉRICA,Estadística,MATG1058,2025,2S,1
7,202407169,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1


In [34]:
df_estudiantes_viendo["CARRERA"].value_counts()

CARRERA
Estadística               20
Logística y Transporte    16
Name: count, dtype: int64

### Prerequisitos necesarios de la materia objetivo (X Correquisitos)

In [35]:
dir_pre_co_requisitos = '../database/planificacion_aprobacion\\corequisito_prerequisito_materias.sql'

In [36]:
# queries SQL parameters
with open(dir_pre_co_requisitos, 'r', encoding='utf-8') as file:
    sql_pre_co_requisitos_base = file.read()


In [37]:
sql_pre_co_requisitos = sql_pre_co_requisitos_base.split("------------")[0]
sql_pre_co_requisitos = sql_pre_co_requisitos.replace('\n', ' ')
sql_pre_co_requisitos = sql_pre_co_requisitos.replace('xxxxx', f"{cod_materia_objetivo}")

In [38]:
sql_pre_co_requisitos

"SELECT distinct pa.nombre carrera, m.codigomateria, m.nombre materia , mm.nivel, m.HORASDOCENCIASEM,m.HORASPRACTICASSEM, m.HORASAUTONOMSEM, m.nombreingles , CASE when m.tipomateria = 'T' then 'teórica' when m.tipomateria = 'I' then 'importada' when m.tipomateria = 'p' then 'practica' when m.tipomateria = 'r' then 'teóricoPractica' when m.tipomateria = 'n' then 'nivelcero' when m.tipomateria = 'l' then 'con laboratorio' when m.tipomateria = 'a' then 'laboratorio' when m.tipomateria = 'g' then 'graduacion' when m.tipomateria = 's' then 'teórico práctico sin laboratorio' when m.tipomateria = 'm' then 'modular' when m.tipomateria = 'c' then 'mención' when m.tipomateria = 'z' then 'paralelo practico unido a teorico x codigomateria'  when m.tipomateria = 'o' then 'integradora' when m.tipomateria = 'v' then 'investigacion' else m.tipomateria end tipomateria, tc.nombre tipocredito , case when m.clasifmateria = 'g' then 'general' when m.clasifmateria = 'c' then 'complementaria' when m.clasifma

In [39]:
# Fetch all rows
data_list = get_data_from_query(sql_pre_co_requisitos)

In [40]:
df_pre_requisito = pd.DataFrame(data_list)[["CARRERA", "MATERIA_REQUISITO", "CODIGOMATERIA", "TIPO", "TIPOMATERIA"]]

In [41]:
df_pre_requisito = df_pre_requisito[df_pre_requisito["TIPO"] == "PR"]

In [42]:
df_pre_requisito.shape

(4, 5)

In [43]:
df_pre_requisito

,CARRERA,MATERIA_REQUISITO,CODIGOMATERIA,TIPO,TIPOMATERIA
0,Estadística,FUNDAMENTOS DE PROGRAMACIÓN,CCPG1043,PR,teóricoPractica
1,Estadística,CÁLCULO VECTORIAL,MATG1046,PR,teóricoPractica
2,Estadística,ÁLGEBRA LINEAL I,MATG1066,PR,teóricoPractica
3,Logística y Transporte,OPTIMIZACIÓN LINEAL,MATG1057,PR,teóricoPractica


### Estudiantes de prerequisitos, cuales han aprobado para considerar en la planificacion (X corequisitos)

In [44]:
lis_carreras_pre = df_pre_requisito["CARRERA"].unique().tolist()

In [45]:
lis_carreras_pre

['Estadística', 'Logística y Transporte']

In [46]:
lis_cod_materias_pre = [i.strip() for i in df_pre_requisito["CODIGOMATERIA"].unique().tolist()]

In [47]:
lis_cod_materias_pre

['CCPG1043', 'MATG1046', 'MATG1066', 'MATG1057']

In [48]:
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_base.split("------------")[0]
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('\n', ' ')
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('TT', f"{termino_base}S")
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('AAAA', f"{anio_base}")
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('xxxxx', "','".join(lis_cod_materias_pre))

In [49]:
sql_estudiantes_viendo_pre

"SELECT COD_ESTUDIANTE, COD_MATERIA_ACAD, tm.NOMBRE, tpa.nombre carrera, hd.*  FROM espol.HISTORIA_ANIO  hd  INNER JOIN espol.TBL_MATERIA tm ON hd.COD_MATERIA_ACAD =tm.CODIGOMATERIA  INNER JOIN espol.TBL_PROGRAMA_ACADEMICO tpa ON tpa.CODCARRERA =hd.COD_CARRERA and tpa.CODDIVISION =hd.COD_DIVISION AND tpa.CODESPECIALIZ =hd.COD_ESPECIALIZ WHERE tm.codigomateria IN ('CCPG1043','MATG1046','MATG1066','MATG1057') AND termino='2S' AND ANIO='2025' AND hd.ESTADO_MAT_TOMADA <>'AN' ;  "

In [50]:
# Fetch all rows
data_list = get_data_from_query(sql_estudiantes_viendo_pre)

In [51]:
df_estudiantes_viendo_pre = pd.DataFrame(data_list)[["COD_ESTUDIANTE", "COD_MATERIA_ACAD", "NOMBRE", "CARRERA", "ANIO", "TERMINO", "VEZ_TOMADA"]].copy()

In [52]:
# hacer antes el copy porque si no no sabe si es en el copy o en el original
df_estudiantes_viendo_pre["COD_MATERIA_ACAD"] = df_estudiantes_viendo_pre["COD_MATERIA_ACAD"].str.strip() 

In [53]:
"Tamaño coincide", df_estudiantes_viendo_pre["COD_ESTUDIANTE"].shape[0] == len(data_list)

('Tamaño coincide', True)

In [54]:
df_estudiantes_viendo_pre = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["CARRERA"].isin(lis_carreras_pre)]

In [55]:
df_estudiantes_viendo_pre["CARRERA"].value_counts()

CARRERA
Estadística               115
Logística y Transporte     63
Name: count, dtype: int64

In [56]:
df_estudiantes_viendo_pre["NOMBRE"].value_counts()

NOMBRE
CÁLCULO VECTORIAL              80
FUNDAMENTOS DE PROGRAMACIÓN    50
ÁLGEBRA LINEAL I               34
OPTIMIZACIÓN LINEAL            14
Name: count, dtype: int64

In [57]:
df_estudiantes_viendo_pre

,COD_ESTUDIANTE,COD_MATERIA_ACAD,NOMBRE,CARRERA,ANIO,TERMINO,VEZ_TOMADA
11,202413910,MATG1066,ÁLGEBRA LINEAL I,Estadística,2025,2S,1
29,202502910,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Logística y Transporte,2025,2S,1
34,202413910,MATG1046,CÁLCULO VECTORIAL,Estadística,2025,2S,1
51,202507984,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Estadística,2025,2S,1
64,202500732,MATG1066,ÁLGEBRA LINEAL I,Estadística,2025,2S,1
...,...,...,...,...,...,...,...
1944,202107702,MATG1066,ÁLGEBRA LINEAL I,Estadística,2025,2S,2
1945,202407813,MATG1046,CÁLCULO VECTORIAL,Estadística,2025,2S,2
1961,202522314,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Logística y Transporte,2025,2S,1
1969,202515516,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Estadística,2025,2S,1


In [58]:
# Estudiantes que no deberian estar nuevamente si se quedan por tercera
list_tercera = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["VEZ_TOMADA"] > 2]["COD_MATERIA_ACAD"]
list_tercera

1222    MATG1046
1474    MATG1046
1581    MATG1046
1804    MATG1046
1814    MATG1066
1845    CCPG1043
Name: COD_MATERIA_ACAD, dtype: object

In [59]:
dict_results = {}
for cod_materia in lis_cod_materias_pre:
    list_matricula = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["COD_MATERIA_ACAD"] == cod_materia]["COD_ESTUDIANTE"].tolist()
    results, mi_df, stats = make_analysis(list_matricula, cod_materia)
    dict_results[cod_materia] = {
        "results": results,
        "dataframe": mi_df,
        "statistics": stats
    }


file_dir ../data/riesgo_academico/saved/inference_data.csv

✅ CCPG1043 encontrado!
   Valor original: CCPG1043, Valor encoded: 151
📊 Matrículas encontradas (50): ['202502910', '202507984', '202503280', '202415436', '202501391', '202506150', '202504809', '202510988', '202519013', '202506002', '202505582', '202413118', '202500492', '202509832', '202500187', '202500856', '202412086', '202505616', '202400255', '202412102', '202503553', '202523098', '202505327', '202416327', '202404455', '202503637', '202416723', '202505111', '202400313', '202415501', '202317392', '202412755', '202511382', '202511820', '202412201', '202309969', '202401709', '202404844', '202206546', '202412250', '202400271', '202515573', '202514758', '202513495', '202516571', '202512257', '202107702', '202522314', '202515516', '202515714']
Debe coinicidir la cantidad: True
📊 Usando DataFrame proporcionado
   ✅ Datos cargados: 50 registros
🔮 Realizando predicciones con 50 registros...
** tmp_categoria_riesgo <class 'pandas.c

c:\Users\saraujo\Documents\Riesgo academico\Codigos_riesgo_academico\planificacion_aprobacion\inference.py:148: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  results["CATEGORIA_RIESGO"] = tmp_categoria_riesgo.to_list()
c:\Users\saraujo\Documents\Riesgo academico\Codigos_riesgo_academico\planificacion_aprobacion\inference.py:148: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  results["CATEGORIA_RIESGO"] = tmp_categoria_riesgo.to_list()



✅ MATG1066 encontrado!
   Valor original: MATG1066, Valor encoded: 435
📊 Matrículas encontradas (34): ['202413910', '202500732', '202507802', '202413050', '202507984', '202510988', '202504809', '202506002', '202508461', '202501771', '202408415', '202506150', '202505582', '202507760', '202311494', '202500187', '202506176', '202506168', '202405890', '202409009', '202506812', '202414231', '202503553', '202006052', '202407763', '202315503', '202403523', '202309969', '202400388', '202416061', '202503637', '202510111', '202206546', '202107702']
Debe coinicidir la cantidad: True
📊 Usando DataFrame proporcionado
   ✅ Datos cargados: 34 registros
🔮 Realizando predicciones con 34 registros...
** tmp_categoria_riesgo <class 'pandas.core.arrays.categorical.Categorical'>
   ✅ Probabilidades calculadas
   ✅ Predicciones completadas
   📊 14 estudiantes aprobarán (41.18%)
   📊 20 estudiantes reprobarán (58.82%)
file_dir ../data/riesgo_academico/saved/inference_data.csv

✅ MATG1057 encontrado!
   Valo

c:\Users\saraujo\Documents\Riesgo academico\Codigos_riesgo_academico\planificacion_aprobacion\inference.py:148: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  results["CATEGORIA_RIESGO"] = tmp_categoria_riesgo.to_list()
c:\Users\saraujo\Documents\Riesgo academico\Codigos_riesgo_academico\planificacion_aprobacion\inference.py:148: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  results["CATEGORIA_RIESGO"] = tmp_categoria_riesgo.to_list()


In [60]:
dict_results.keys()

dict_keys(['CCPG1043', 'MATG1046', 'MATG1066', 'MATG1057'])

In [61]:
df_estudiantes_viendo_pre_tmp = df_estudiantes_viendo_pre.copy()
df_estudiantes_viendo_pre_tmp["APROBADO"] = 0

In [62]:
for cod_materia in lis_cod_materias_pre:
    list_matricula = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["COD_MATERIA_ACAD"] == cod_materia]["COD_ESTUDIANTE"].tolist()
    list_result = dict_results[cod_materia]["results"]["predictions"]
    indice_result = 0
    for i in list_matricula:
        df_estudiantes_viendo_pre_tmp.loc[(df_estudiantes_viendo_pre_tmp["COD_ESTUDIANTE"] == i) & (df_estudiantes_viendo_pre_tmp["COD_MATERIA_ACAD"] == cod_materia), "APROBADO"] = list_result[indice_result]
        indice_result += 1  

In [63]:
df_estudiantes_viendo_pre.shape, df_estudiantes_viendo_pre_tmp.shape, df_estudiantes_viendo_pre_tmp["APROBADO"].value_counts()

((178, 7),
 (178, 8),
 APROBADO
 0    125
 1     53
 Name: count, dtype: int64)

In [64]:
df_estudiantes_viendo_pre_tmp

,COD_ESTUDIANTE,COD_MATERIA_ACAD,NOMBRE,CARRERA,ANIO,TERMINO,VEZ_TOMADA,APROBADO
11,202413910,MATG1066,ÁLGEBRA LINEAL I,Estadística,2025,2S,1,1
29,202502910,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Logística y Transporte,2025,2S,1,0
34,202413910,MATG1046,CÁLCULO VECTORIAL,Estadística,2025,2S,1,1
51,202507984,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Estadística,2025,2S,1,1
64,202500732,MATG1066,ÁLGEBRA LINEAL I,Estadística,2025,2S,1,1
...,...,...,...,...,...,...,...,...
1944,202107702,MATG1066,ÁLGEBRA LINEAL I,Estadística,2025,2S,2,0
1945,202407813,MATG1046,CÁLCULO VECTORIAL,Estadística,2025,2S,2,0
1961,202522314,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Logística y Transporte,2025,2S,1,0
1969,202515516,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Estadística,2025,2S,1,0


### Analizar los estudiantes actuales (2025-2S) con el modelo actual, para quedarnos con los reprobados para la planificacion 2026


In [65]:
list_matricula = df_estudiantes_viendo["COD_ESTUDIANTE"].tolist()
len(list_matricula), df_estudiantes_viendo["COD_ESTUDIANTE"].nunique()

(36, 36)

In [66]:
df_estudiantes_viendo_tmp = df_estudiantes_viendo.copy()
df_estudiantes_viendo_tmp["APROBADO"] = 0

In [67]:
results, mi_df, stats = make_analysis(list_matricula, cod_materia_objetivo)

file_dir ../data/riesgo_academico/saved/inference_data.csv

✅ MATG1058 encontrado!
   Valor original: MATG1058, Valor encoded: 427
📊 Matrículas encontradas (36): ['202003976', '202317301', '202416137', '202407136', '202317137', '202306775', '202407318', '202407169', '202407847', '202405494', '202315461', '202213070', '202310488', '202400735', '202407995', '202406849', '202400958', '202316600', '202401683', '201706215', '202302394', '202105672', '202403473', '202404158', '202210720', '202318960', '201904703', '202107017', '202300299', '202403903', '202312278', '202213039', '202309399', '202408324', '202300430', '202400701']
Debe coinicidir la cantidad: True
📊 Usando DataFrame proporcionado
   ✅ Datos cargados: 36 registros
🔮 Realizando predicciones con 36 registros...
** tmp_categoria_riesgo <class 'pandas.core.arrays.categorical.Categorical'>
   ✅ Probabilidades calculadas
   ✅ Predicciones completadas
   📊 11 estudiantes aprobarán (30.56%)
   📊 25 estudiantes reprobarán (69.44%)


c:\Users\saraujo\Documents\Riesgo academico\Codigos_riesgo_academico\planificacion_aprobacion\inference.py:148: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  results["CATEGORIA_RIESGO"] = tmp_categoria_riesgo.to_list()


In [68]:
index_matricula = 0
for i in list_matricula:
    # print("Procesando estudiante:", i, "Índice:", index_matricula)
    df_estudiantes_viendo_tmp.loc[(df_estudiantes_viendo_tmp["COD_ESTUDIANTE"] == i), "APROBADO"] = results["predictions"][index_matricula]
    index_matricula += 1

In [69]:
df_estudiantes_viendo.shape, df_estudiantes_viendo_tmp.shape, df_estudiantes_viendo_tmp["APROBADO"].value_counts()

((36, 7),
 (36, 8),
 APROBADO
 0    25
 1    11
 Name: count, dtype: int64)

### Estudiantes que ya han visto las prerequisitos pero no la objetivo y la materia objetivo. Y les tocaria ver la materia objetivo en 2026-1S

In [70]:
df_estudiantes_viendo_pre_tmp

,COD_ESTUDIANTE,COD_MATERIA_ACAD,NOMBRE,CARRERA,ANIO,TERMINO,VEZ_TOMADA,APROBADO
11,202413910,MATG1066,ÁLGEBRA LINEAL I,Estadística,2025,2S,1,1
29,202502910,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Logística y Transporte,2025,2S,1,0
34,202413910,MATG1046,CÁLCULO VECTORIAL,Estadística,2025,2S,1,1
51,202507984,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Estadística,2025,2S,1,1
64,202500732,MATG1066,ÁLGEBRA LINEAL I,Estadística,2025,2S,1,1
...,...,...,...,...,...,...,...,...
1944,202107702,MATG1066,ÁLGEBRA LINEAL I,Estadística,2025,2S,2,0
1945,202407813,MATG1046,CÁLCULO VECTORIAL,Estadística,2025,2S,2,0
1961,202522314,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Logística y Transporte,2025,2S,1,0
1969,202515516,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,Estadística,2025,2S,1,0


In [71]:
df_estudiantes_viendo_tmp

,COD_ESTUDIANTE,NOMBRE,CARRERA,COD_MATERIA_ACAD,ANIO,TERMINO,VEZ_TOMADA,APROBADO
0,202003976,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1,1
1,202317301,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1,1
2,202416137,OPTIMIZACIÓN NUMÉRICA,Estadística,MATG1058,2025,2S,1,1
3,202407136,OPTIMIZACIÓN NUMÉRICA,Estadística,MATG1058,2025,2S,1,0
4,202317137,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1,0
5,202306775,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1,0
6,202407318,OPTIMIZACIÓN NUMÉRICA,Estadística,MATG1058,2025,2S,1,1
7,202407169,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1,1
8,202407847,OPTIMIZACIÓN NUMÉRICA,Estadística,MATG1058,2025,2S,1,0
9,202405494,OPTIMIZACIÓN NUMÉRICA,Logística y Transporte,MATG1058,2025,2S,1,1


In [72]:
lis_cod_materias_pre, lis_carreras_pre, df_pre_requisito["CARRERA"].unique().tolist(), cod_materia_objetivo

(['CCPG1043', 'MATG1046', 'MATG1066', 'MATG1057'],
 ['Estadística', 'Logística y Transporte'],
 ['Estadística', 'Logística y Transporte'],
 'MATG1058')

In [73]:
df_pre_requisito

,CARRERA,MATERIA_REQUISITO,CODIGOMATERIA,TIPO,TIPOMATERIA
0,Estadística,FUNDAMENTOS DE PROGRAMACIÓN,CCPG1043,PR,teóricoPractica
1,Estadística,CÁLCULO VECTORIAL,MATG1046,PR,teóricoPractica
2,Estadística,ÁLGEBRA LINEAL I,MATG1066,PR,teóricoPractica
3,Logística y Transporte,OPTIMIZACIÓN LINEAL,MATG1057,PR,teóricoPractica


##### Descartar los RP y que estan viendo por tercera vez porque ya perdieron la carrera


In [74]:
list_tercera_and_rp = df_estudiantes_viendo_pre_tmp[(df_estudiantes_viendo_pre_tmp["APROBADO"] == 0) & (df_estudiantes_viendo_pre_tmp["VEZ_TOMADA"] > 2)]["COD_ESTUDIANTE"].unique().tolist()
df_estudiantes_viendo_tmp = df_estudiantes_viendo_tmp[~df_estudiantes_viendo_tmp["COD_ESTUDIANTE"].isin(list_tercera_and_rp)]

In [75]:
df_estudiantes_viendo_tmp.shape, df_estudiantes_viendo_tmp["APROBADO"].value_counts()

((36, 8),
 APROBADO
 0    25
 1    11
 Name: count, dtype: int64)

In [76]:
df_complete["termino"].value_counts()

termino
1S    185377
2S    181523
Name: count, dtype: int64

##### Actualizar la columna ESTADO_MAT_TOMADA_MO en df_complete

In [77]:
# materias de pre requisito, periodo actual, estudiantes viendo esas materias
df_complete[(df_complete["COD_MATERIA_ACAD_MO"].isin(lis_cod_materias_pre)) & (df_complete["anio"] == str(anio_base)) & (df_complete["termino"] == str(termino_base)+"S") & (df_complete["COD_ESTUDIANTE"].isin(df_estudiantes_viendo_pre_tmp["COD_ESTUDIANTE"]))]

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MODERADA,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA
341245,202107702,CCPG1043,AC,2,0,0,"0,00","5,53",21.0,64.0,...,0,0,3,NaN,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,2025,2S,"7,38",Estadística
341249,202206546,CCPG1043,AC,2,0,0,"0,00","5,53",26.0,51.0,...,0,0,3,NaN,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,2025,2S,"7,22",Estadística
341263,202502910,CCPG1043,AC,1,0,0,"0,00","5,53",3.0,47.0,...,0,0,4,NaN,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,2025,2S,"9,00",Logística y Transporte
341282,202514758,CCPG1043,AC,1,0,0,"0,00","5,53",NaN,NaN,...,0,1,3,NaN,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,2025,2S,"0,00",Logística y Transporte
341290,202522314,CCPG1043,AC,1,0,0,"0,00","5,53",NaN,NaN,...,0,1,2,NaN,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN,2025,2S,"0,00",Logística y Transporte
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360480,202507802,MATG1066,AC,1,0,0,"0,00","5,10",4.0,90.0,...,0,0,3,NaN,MATG1066,ÁLGEBRA LINEAL I,2025,2S,"8,37",Estadística
360481,202409009,MATG1066,AC,1,0,0,"0,00","5,10",12.0,77.0,...,0,0,4,NaN,MATG1066,ÁLGEBRA LINEAL I,2025,2S,"8,08",Estadística
360482,202506176,MATG1066,AC,1,0,0,"0,00","5,10",4.0,65.0,...,0,0,3,NaN,MATG1066,ÁLGEBRA LINEAL I,2025,2S,"7,90",Estadística
360483,202506150,MATG1066,AC,1,0,0,"0,00","5,10",4.0,85.0,...,0,0,3,NaN,MATG1066,ÁLGEBRA LINEAL I,2025,2S,"8,21",Estadística


In [78]:
# Mapeo de valores de APROBADO
mapping = {1: 'AP', 0: 'RP'}

In [79]:
# Actualizar df_complete
for index, row in df_estudiantes_viendo_pre_tmp.iterrows():
    # print("Procesando estudiante:", row['COD_ESTUDIANTE'], "Materia:", row['COD_MATERIA_ACAD'], "Aprobado:", row['APROBADO'])
    df_complete.loc[(df_complete['COD_ESTUDIANTE'] == row['COD_ESTUDIANTE']) & 
                    (df_complete['COD_MATERIA_ACAD_MO'] == row['COD_MATERIA_ACAD']) & 
                    (df_complete['anio'] == str(anio_base)) & 
                    (df_complete['termino'] == str(termino_base)+"S"), 
                    'ESTADO_MAT_TOMADA_MO'] = mapping[row['APROBADO']]  

In [80]:
# df_complete[(df_complete["COD_ESTUDIANTE"] == '202515714') & (df_complete["COD_MATERIA_ACAD_MO"] == 'CCPG1043')]

In [81]:
# Actualizar df_complete
for index, row in df_estudiantes_viendo_tmp.iterrows():
    # print("Procesando estudiante:", row['COD_ESTUDIANTE'], "Materia:", row['COD_MATERIA_ACAD'], "Aprobado:", row['APROBADO'])
    df_complete.loc[(df_complete['COD_ESTUDIANTE'] == row['COD_ESTUDIANTE']) & 
                    (df_complete['COD_MATERIA_ACAD_MO'] == row['COD_MATERIA_ACAD']) & 
                    (df_complete['anio'] == str(anio_base)) & 
                    (df_complete['termino'] == str(termino_base)+"S"), 
                    'ESTADO_MAT_TOMADA_MO'] = mapping[row['APROBADO']]  

In [82]:
# df_complete[(df_complete["COD_ESTUDIANTE"] == '202400701') & (df_complete["COD_MATERIA_ACAD_MO"] == 'MATG1058')]

##### Obtener los estudiantes que veran la materia objetivo en 2026-1S

In [83]:
df_complete

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MODERADA,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA
0,201160178,ACUG1035,AP,1,87,93,"9,00","7,90",53.0,59.0,...,2,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura
1,201310353,ACUG1035,AP,1,85,89,"8,70","7,90",59.0,61.0,...,2,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura
2,201313869,ACUG1035,AP,1,86,89,"8,75","7,90",59.0,56.0,...,1,0,1,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura
3,201507649,ACUG1035,AP,1,83,93,"8,80","7,90",49.0,65.0,...,2,2,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura
4,201607884,ACUG1035,AP,1,95,95,"9,50","7,90",39.0,72.0,...,1,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366895,202300422,TURG2037,AC,1,0,0,"0,00","7,22",27.0,73.0,...,0,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,92",Turismo
366896,202104683,TURG2037,AC,1,0,0,"0,00","7,22",37.0,65.0,...,0,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,54",Turismo
366897,202111548,TURG2037,AC,1,0,0,"0,00","7,22",36.0,66.0,...,0,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,53",Turismo
366898,202103719,TURG2037,AC,1,0,0,"0,00","7,22",33.0,67.0,...,0,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,25",Turismo


### 1. Repitentes: Estudiantes con último estado RP o PF en la materia objetivo

In [84]:
# Filtrar estudiantes que han cursado la materia objetivo
df_cursaron_objetivo = df_complete[df_complete['COD_MATERIA_ACAD_MO'] == cod_materia_objetivo].copy()

# Verificar si alguna vez aprobaron (AP) la materia objetivo
estudiantes_con_ap = df_cursaron_objetivo[df_cursaron_objetivo['ESTADO_MAT_TOMADA_MO'] == 'AP']['COD_ESTUDIANTE'].unique()

# Filtrar solo estudiantes que nunca aprobaron (excluir los que tienen AP)
df_sin_ap = df_cursaron_objetivo[~df_cursaron_objetivo['COD_ESTUDIANTE'].isin(estudiantes_con_ap)]

# Filtrar solo RP o PF
df_rp_pf = df_sin_ap[df_sin_ap['ESTADO_MAT_TOMADA_MO'].isin(['RP', 'PF'])].copy()
print("Deberian ser igual: ", df_sin_ap.shape[0] == df_rp_pf.shape[0])

# Ordenar por estudiante y fecha para obtener el último registro
df_rp_pf['termino_num'] = df_rp_pf['termino'].str.replace('S', '').astype(int)
df_rp_pf = df_rp_pf.sort_values(['anio', 'termino_num'], ascending=[True, True])

# Obtener el último registro por estudiante
df_repitentes = df_rp_pf.groupby('COD_ESTUDIANTE').first().reset_index()

Deberian ser igual:  True


In [85]:
print(f"Total de repitentes: {df_repitentes.shape[0]}")
df_repitentes.head()

Total de repitentes: 36


,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num
0,200919637,MATG1058,RP,1,3,0,"0,34","5,85",38.0,48.0,...,0,3,None,MATG1058,OPTIMIZACIÓN NUMÉRICA,2020,1S,None,Logística y Transporte,1
1,201247810,MATG1058,RP,1,45,43,"5,73","5,85",41.0,50.0,...,0,4,None,MATG1058,OPTIMIZACIÓN NUMÉRICA,2021,2S,None,Estadística,2
2,201303783,MATG1058,PF,1,0,0,"0,00","5,85",64.0,47.0,...,0,4,None,MATG1058,OPTIMIZACIÓN NUMÉRICA,2021,2S,None,Logística y Transporte,2
3,201405497,MATG1058,RP,1,31,29,"3,30","5,85",43.0,51.0,...,0,4,None,MATG1058,OPTIMIZACIÓN NUMÉRICA,2022,1S,None,Logística y Transporte,1
4,201606050,MATG1058,RP,1,47,37,"4,89","5,85",36.0,48.0,...,0,4,None,MATG1058,OPTIMIZACIÓN NUMÉRICA,2020,2S,None,Estadística,2


In [86]:
df_repitentes.groupby(["anio", "termino"]).size()

anio  termino
2020  1S          3
      2S          1
2021  1S          1
      2S          2
2022  1S          2
2024  1S          1
2025  1S          1
      2S         25
dtype: int64

### 2. Nuevos: Estudiantes que cumplen prerrequisitos y nunca cursaron la materia objetivo

In [119]:
# 1. Obtener las materias prerrequisito por carrera
prereq_por_carrera = (
    df_pre_requisito.groupby('CARRERA')['CODIGOMATERIA']
    .apply(lambda x: set(x.str.strip()))
    .reset_index()
    .rename(columns={'CODIGOMATERIA': 'materias_requeridas'})
)

In [121]:
# 2. Filtrar registros de df_complete donde el estudiante APROBÓ los prerrequisitos
df_complete_prereq_aprobados = df_complete[
    (df_complete['COD_MATERIA_ACAD_MO'].isin(lis_cod_materias_pre)) &
    (df_complete['CARRERA'].isin(lis_carreras_pre)) &
    (df_complete['ESTADO_MAT_TOMADA_MO'] == 'AP')  # Solo materias aprobadas
].copy()

In [125]:
# 3. Agrupar por estudiante y carrera para obtener materias aprobadas
materias_aprobadas_por_estudiante = (
    df_complete_prereq_aprobados.groupby(['COD_ESTUDIANTE', 'CARRERA'])['COD_MATERIA_ACAD_MO']
    .apply(lambda x: set(x.str.strip() if hasattr(x, 'str') else x))
    .reset_index()
    .rename(columns={'COD_MATERIA_ACAD_MO': 'materias_aprobadas'})
)

In [141]:
materias_aprobadas_por_estudiante


,COD_ESTUDIANTE,CARRERA,materias_aprobadas
0,201133318,Logística y Transporte,{MATG1057}
1,201154793,Estadística,"{CCPG1043, MATG1046}"
2,201164839,Logística y Transporte,{MATG1057}
3,201204176,Logística y Transporte,"{MATG1057, CCPG1043, MATG1046}"
4,201229676,Logística y Transporte,"{MATG1057, CCPG1043, MATG1046}"
...,...,...,...
340,202508461,Estadística,"{MATG1066, CCPG1043, MATG1046}"
341,202509832,Logística y Transporte,{MATG1046}
342,202510988,Estadística,"{MATG1066, MATG1046}"
343,202515573,Estadística,{CCPG1043}


In [127]:
# 4. Hacer merge para comparar con prerrequisitos requeridos
df_comparacion = pd.merge(
    materias_aprobadas_por_estudiante,
    prereq_por_carrera,
    on='CARRERA',
    how='inner'
)

In [129]:
# 5. Verificar que tengan TODOS los prerrequisitos aprobados
df_comparacion['tiene_todos_prereq'] = df_comparacion.apply(
    lambda row: row['materias_requeridas'].issubset(row['materias_aprobadas']),
    axis=1
)

estudiantes_con_prereq_completos = df_comparacion[df_comparacion['tiene_todos_prereq']]

In [131]:
# 6. Filtrar estudiantes que NUNCA han visto la materia objetivo
estudiantes_vieron_objetivo = set(
    df_complete[df_complete['COD_MATERIA_ACAD_MO'] == cod_materia_objetivo]['COD_ESTUDIANTE']
)

In [133]:
# 7. Resultado final: estudiantes elegibles (con prereq aprobados y sin haber visto objetivo)
df_nuevos = estudiantes_con_prereq_completos[
    ~estudiantes_con_prereq_completos['COD_ESTUDIANTE'].isin(estudiantes_vieron_objetivo)
].copy()

In [134]:
print(f"📊 Resumen:")
print(f"  - Estudiantes con todos los prerrequisitos APROBADOS: {len(estudiantes_con_prereq_completos)}")
print(f"  - Estudiantes que vieron {cod_materia_objetivo}: {len(estudiantes_vieron_objetivo)}")
print(f"  - Estudiantes NUEVOS elegibles: {len(df_nuevos)}")
print(f"\nDistribución por carrera:")
print(df_nuevos['CARRERA'].value_counts())

# Ver resultado
df_nuevos[['COD_ESTUDIANTE', 'CARRERA', 'materias_aprobadas', 'materias_requeridas']]

📊 Resumen:
  - Estudiantes con todos los prerrequisitos APROBADOS: 230
  - Estudiantes que vieron MATG1058: 227
  - Estudiantes NUEVOS elegibles: 53

Distribución por carrera:
CARRERA
Logística y Transporte    32
Estadística               21
Name: count, dtype: int64


,COD_ESTUDIANTE,CARRERA,materias_aprobadas,materias_requeridas
4,201229676,Logística y Transporte,"{MATG1057, CCPG1043, MATG1046}",{MATG1057}
7,201314045,Logística y Transporte,{MATG1057},{MATG1057}
9,201405682,Logística y Transporte,{MATG1057},{MATG1057}
13,201503184,Logística y Transporte,{MATG1057},{MATG1057}
14,201503899,Logística y Transporte,{MATG1057},{MATG1057}
15,201504025,Logística y Transporte,{MATG1057},{MATG1057}
16,201602380,Logística y Transporte,{MATG1057},{MATG1057}
18,201608817,Logística y Transporte,{MATG1057},{MATG1057}
20,201612520,Logística y Transporte,"{MATG1057, CCPG1043, MATG1046}",{MATG1057}
21,201612538,Logística y Transporte,{MATG1057},{MATG1057}


In [137]:
# 1. Validación final: Verificar que TODOS los prerrequisitos estén cumplidos
print("🔍 Validación de prerrequisitos por estudiante:")
for idx, row in df_nuevos.iterrows():
    faltantes = row['materias_requeridas'] - row['materias_aprobadas']
    if len(faltantes) > 0:
        print(f"  ⚠️ Estudiante {row['COD_ESTUDIANTE']} - Carrera: {row['CARRERA']} - Faltantes: {faltantes}")

# Verificar que no haya faltantes
df_nuevos['prereq_completos'] = df_nuevos.apply(
    lambda row: len(row['materias_requeridas'] - row['materias_aprobadas']) == 0,
    axis=1
)
print(f"\n✅ Todos cumplen prerrequisitos: {df_nuevos['prereq_completos'].all()}")
print(f"Total estudiantes con prerrequisitos completos: {df_nuevos['prereq_completos'].sum()}")

🔍 Validación de prerrequisitos por estudiante:

✅ Todos cumplen prerrequisitos: True
Total estudiantes con prerrequisitos completos: 53


In [144]:
# 2. Obtener información adicional de cada estudiante desde df_complete
# Para cada estudiante, obtener su último registro (año y término más reciente)
df_complete_temp = df_complete.copy()
df_complete_temp['termino_num'] = df_complete_temp['termino'].str.replace('S', '').astype(int)
df_complete_temp = df_complete_temp.sort_values(['anio', 'termino_num'], ascending=[False, False])

# Obtener la última fila por estudiante
df_ultimos_registros = df_complete_temp.groupby('COD_ESTUDIANTE').first().reset_index()

# Hacer merge con df_nuevos para agregar información del último periodo
df_nuevos_completo = pd.merge(
    df_nuevos[['COD_ESTUDIANTE', 'CARRERA', 'materias_aprobadas', 'materias_requeridas']],
    df_ultimos_registros[['COD_ESTUDIANTE', 'anio', 'termino', 'COD_MATERIA_ACAD_MO', 'MATERIA']],
    on='COD_ESTUDIANTE',
    how='left'
)

print(f"📋 Estudiantes nuevos con información del último periodo:")
print(f"Total: {len(df_nuevos_completo)}")
print(f"\nDistribución por año y término del último registro:")
print(df_nuevos_completo.groupby(['anio', 'termino']).size().sort_index(ascending=False))

df_nuevos_completo

📋 Estudiantes nuevos con información del último periodo:
Total: 53

Distribución por año y término del último registro:
anio  termino
2025  2S         32
      1S          1
2023  2S          1
      1S          2
2022  2S          4
      1S          7
2021  2S          4
2020  1S          2
dtype: int64


,COD_ESTUDIANTE,CARRERA,materias_aprobadas,materias_requeridas,anio,termino,COD_MATERIA_ACAD_MO,MATERIA
0,201229676,Logística y Transporte,"{MATG1057, CCPG1043, MATG1046}",{MATG1057},2022,2S,FISG1005,FÍSICA: MECÁNICA
1,201314045,Logística y Transporte,{MATG1057},{MATG1057},2021,2S,LOGG1023,PRESUPUESTOS Y PROYECTOS LOGÍSTICOS
2,201405682,Logística y Transporte,{MATG1057},{MATG1057},2020,1S,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN
3,201503184,Logística y Transporte,{MATG1057},{MATG1057},2021,2S,ESTG1051,SERIES DE TIEMPO
4,201503899,Logística y Transporte,{MATG1057},{MATG1057},2022,2S,MATG1061,METAHEURÍSTICAS
5,201504025,Logística y Transporte,{MATG1057},{MATG1057},2022,2S,MATG1061,METAHEURÍSTICAS
6,201602380,Logística y Transporte,{MATG1057},{MATG1057},2020,1S,LOGG1016,TRANSPORTE MULTIMODAL
7,201608817,Logística y Transporte,{MATG1057},{MATG1057},2022,1S,ADMG1007,ESTRATEGIAS DE GESTIÓN
8,201612520,Logística y Transporte,"{MATG1057, CCPG1043, MATG1046}",{MATG1057},2021,2S,ADMG1005,EMPRENDIMIENTO E INNOVACIÓN
9,201612538,Logística y Transporte,{MATG1057},{MATG1057},2023,2S,ADMG1007,ESTRATEGIAS DE GESTIÓN


In [145]:
# Renombrar df_nuevos_completo a df_nuevos para mantener consistencia
df_nuevos = df_nuevos_completo.copy()

# Verificación final
print(f"\n✅ DataFrame df_nuevos actualizado con:")
print(f"  - Validación de prerrequisitos completos")
print(f"  - Información del último año y término por estudiante")
print(f"\nColumnas disponibles: {list(df_nuevos.columns)}")


✅ DataFrame df_nuevos actualizado con:
  - Validación de prerrequisitos completos
  - Información del último año y término por estudiante

Columnas disponibles: ['COD_ESTUDIANTE', 'CARRERA', 'materias_aprobadas', 'materias_requeridas', 'anio', 'termino', 'COD_MATERIA_ACAD_MO', 'MATERIA']


In [146]:
df_nuevos

,COD_ESTUDIANTE,CARRERA,materias_aprobadas,materias_requeridas,anio,termino,COD_MATERIA_ACAD_MO,MATERIA
0,201229676,Logística y Transporte,"{MATG1057, CCPG1043, MATG1046}",{MATG1057},2022,2S,FISG1005,FÍSICA: MECÁNICA
1,201314045,Logística y Transporte,{MATG1057},{MATG1057},2021,2S,LOGG1023,PRESUPUESTOS Y PROYECTOS LOGÍSTICOS
2,201405682,Logística y Transporte,{MATG1057},{MATG1057},2020,1S,CCPG1043,FUNDAMENTOS DE PROGRAMACIÓN
3,201503184,Logística y Transporte,{MATG1057},{MATG1057},2021,2S,ESTG1051,SERIES DE TIEMPO
4,201503899,Logística y Transporte,{MATG1057},{MATG1057},2022,2S,MATG1061,METAHEURÍSTICAS
5,201504025,Logística y Transporte,{MATG1057},{MATG1057},2022,2S,MATG1061,METAHEURÍSTICAS
6,201602380,Logística y Transporte,{MATG1057},{MATG1057},2020,1S,LOGG1016,TRANSPORTE MULTIMODAL
7,201608817,Logística y Transporte,{MATG1057},{MATG1057},2022,1S,ADMG1007,ESTRATEGIAS DE GESTIÓN
8,201612520,Logística y Transporte,"{MATG1057, CCPG1043, MATG1046}",{MATG1057},2021,2S,ADMG1005,EMPRENDIMIENTO E INNOVACIÓN
9,201612538,Logística y Transporte,{MATG1057},{MATG1057},2023,2S,ADMG1007,ESTRATEGIAS DE GESTIÓN


### 3. Unión: DataFrame final con Repitentes y Nuevos

In [147]:
# Agregar columna identificadora del tipo de estudiante
df_repitentes['TIPO_ESTUDIANTE'] = 'REPITENTE'
df_nuevos['TIPO_ESTUDIANTE'] = 'NUEVO'

# Unir ambos dataframes
df_final = pd.concat([df_repitentes, df_nuevos], ignore_index=True)

# Ordenar por tipo y código de estudiante
df_final = df_final.sort_values(['TIPO_ESTUDIANTE', 'COD_ESTUDIANTE']).reset_index(drop=True)

print(f"\n📊 RESUMEN FINAL:")
print(f"  • Repitentes: {df_repitentes.shape[0]}")
print(f"  • Nuevos: {df_nuevos.shape[0]}")
print(f"  • TOTAL: {df_final.shape[0]}")
print(f"\nDistribución por tipo:")
print(df_final['TIPO_ESTUDIANTE'].value_counts())

df_final


📊 RESUMEN FINAL:
  • Repitentes: 36
  • Nuevos: 53
  • TOTAL: 89

Distribución por tipo:
TIPO_ESTUDIANTE
NUEVO        53
REPITENTE    36
Name: count, dtype: int64


,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,TIPO_ESTUDIANTE,materias_aprobadas,materias_requeridas
0,201229676,FISG1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,FÍSICA: MECÁNICA,2022,2S,NaN,Logística y Transporte,NaN,NUEVO,"{MATG1057, CCPG1043, MATG1046}",{MATG1057}
1,201314045,LOGG1023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,PRESUPUESTOS Y PROYECTOS LOGÍSTICOS,2021,2S,NaN,Logística y Transporte,NaN,NUEVO,{MATG1057},{MATG1057}
2,201405682,CCPG1043,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,FUNDAMENTOS DE PROGRAMACIÓN,2020,1S,NaN,Logística y Transporte,NaN,NUEVO,{MATG1057},{MATG1057}
3,201503184,ESTG1051,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,SERIES DE TIEMPO,2021,2S,NaN,Logística y Transporte,NaN,NUEVO,{MATG1057},{MATG1057}
4,201503899,MATG1061,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,METAHEURÍSTICAS,2022,2S,NaN,Logística y Transporte,NaN,NUEVO,{MATG1057},{MATG1057}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,202403903,MATG1058,RP,1.0,0.0,0.0,"0,00","5,85",10.0,57.0,...,MATG1058,OPTIMIZACIÓN NUMÉRICA,2025,2S,"7,50",Estadística,2.0,REPITENTE,NaN,NaN
85,202404158,MATG1058,RP,1.0,0.0,0.0,"0,00","5,85",11.0,57.0,...,MATG1058,OPTIMIZACIÓN NUMÉRICA,2025,2S,"7,12",Estadística,2.0,REPITENTE,NaN,NaN
86,202407136,MATG1058,RP,1.0,0.0,0.0,"0,00","5,85",12.0,67.0,...,MATG1058,OPTIMIZACIÓN NUMÉRICA,2025,2S,"7,79",Estadística,2.0,REPITENTE,NaN,NaN
87,202407847,MATG1058,RP,1.0,0.0,0.0,"0,00","5,85",14.0,65.0,...,MATG1058,OPTIMIZACIÓN NUMÉRICA,2025,2S,"7,51",Estadística,2.0,REPITENTE,NaN,NaN


In [ ]:
# Verificación: mostrar ejemplos de cada tipo
print("\n🔍 Ejemplos de REPITENTES:")
print(df_final[df_final['TIPO_ESTUDIANTE'] == 'REPITENTE'][['COD_ESTUDIANTE', 'TIPO_ESTUDIANTE', 'ESTADO_MAT_TOMADA_MO', 'anio', 'termino']].head())

print("\n🔍 Ejemplos de NUEVOS:")
print(df_final[df_final['TIPO_ESTUDIANTE'] == 'NUEVO'][['COD_ESTUDIANTE', 'TIPO_ESTUDIANTE']].head())